In [14]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from dotenv import load_dotenv
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated, Literal
import asyncio

load_dotenv()

True

In [15]:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

In [16]:
# search tool
search_tool = DuckDuckGoSearchRun(
    name="Internet_search",
    description=(
        "should search updated info from the internet"
        "use this tool when user ask about current events"
        "news, current information, information require"
        "use internet for search"
    )
)

In [17]:
#custom tool
@tool
def calculate(first_num: float, second_num: float, operation: str)-> dict:
    """
        perform the arithametic operation on the two numbers
        operations allowed are add, sub, multi, div
    """
    try:
            if operation == "add":
                result = first_num + second_num
            elif operation == "sub":
                result = first_num - second_num
            elif operation == "multi":
                result = first_num * second_num
            elif operation == "div":
                if second_num == 0:
                    return {"error": "ZeroDivisionError please try with number"}
                result = first_num/second_num
            else:
                return {"error": "Invalid operation please try with add, sub, multi, div"}
            
            return {"first_num": first_num, "second_num": second_num, "operation": operation, "result": result}
        
    except Exception as e:
            return {"error": str(e)}

In [18]:
# tool combining 
tools = [search_tool, calculate]

# tool binding
llm_with_tool = llm.bind_tools(tools)

In [20]:
# now lets create a state for chatbot
class chatstate(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [24]:
# we will build a function for building graph
def build_graph():
    
    # lets create out node
    async def chat_node(state: chatstate):
        """LLM node that may answer or request a tool call"""
        messages = state['messages']
        response = await llm_with_tool.ainvoke(messages)  # await and async invoke written as(ainvoke)
        return {'messages': [response]}
    
    # our tool node
    tool_node = ToolNode(tools) # toolnode dont need async since its implementation is async internally.
    
    
    # now lets create our graph
    graph = StateGraph(chatstate)
    
    graph.add_node("chat_node", chat_node)
    graph.add_node("tools", tool_node)
    
    graph.add_edge(START, "chat_node")
    graph.add_conditional_edges("chat_node", tools_condition)
    
    graph.add_edge("tools", "chat_node")
    
    chatbot = graph.compile()
    
    return chatbot

In [25]:
async def main():
    # we will get chatbot from build_graph_function
    chatbot = build_graph()
    
    # the response also with ainvoke
    response = await chatbot.ainvoke({'messages': [HumanMessage(content="write 10 lines about ancient egypt")]})
    print(response['messages'][-1].content)
    
# if __name__ == "__main__":
#     asyncio.run(main()) # this yu cannot use since this is ipynb file if its a .py file it must have workd
# so use
await main()

Ancient Egypt was a civilization that thrived along the Nile River in northeastern Africa.
The pyramids of Giza, built around 2580 BC, are one of the most famous landmarks of Ancient Egypt.
The Great Pyramid of Giza, also known as the Pyramid of Khufu, is the largest of the three pyramids.
Ancient Egyptian society was divided into social classes, with the pharaoh at the top and slaves at the bottom.
The pharaohs were believed to be gods on earth, and they held absolute power over the people.
The Ancient Egyptians developed a system of hieroglyphics, which was used to write and record important events.
Mummification was a common practice in Ancient Egypt, where the body was preserved to ensure the person's soul could return to it in the afterlife.
The Nile River played a crucial role in the development of Ancient Egyptian civilization, providing water and fertile soil for farming.
Ancient Egyptian architecture is characterized by the use of stone, particularly limestone and granite, and